# Notebook to process lysosomal and mitochondrial morphology from cellprofiler features - Playground / troubleshooting nb
Inputs Required
 - Database with cellprofiler outputs
   - Mainly use the Per_Cell table for per-cell features
     - Intensity
     - AreaShape features
     - Ratio of organelle area to total cell area
     - Texture features
     - Granularity features
     - Radial intensity distribution about nucleus
   - May additionally want to use mitochdondria or lysosomes.csv if analyzing these specifically 
     - Note that lysosome segmentation is not perfect (but i'm proud of it)
 - Metadata CSV representing 96-well platemap
   - Long-form table containing passage number, staining conditons, treatments, etc
   - Produced automatically from an 8x12 table with the 96Well_PlateMap code
Outputs
- graphs for individual features
- clustering
## Imports

In [ ]:
#Imports
import os
import numpy as np
import pandas as pd
import sqlite3
#plotting
import plotly.express as px
import plotly.graph_objects as go
import matplotlib.pyplot as plt
%matplotlib inline
import seaborn as sns

#import joypy
from scipy import stats

from plate_information import *
from plate_preprocessing import *


## Test area for a single tablee

In [55]:
db_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v4_active/output.db"
extra_db_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v4_active/extra_features.db"
conn = sqlite3.connect(db_path)
cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
nucleus_df = pd.read_sql_query("SELECT * FROM Per_Nuclei", conn)
cyto_df = pd.read_sql_query("SELECT * FROM Per_Cytoplasm", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
cursor = conn.cursor()

combo_df = cell_df.merge(image_df, on=["ImageNumber"], how="left")
combo_df.columns = combo_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)

#display(combo_df[combo_df["Metadata_Well"] == "B01"])
#note the 0328 seems to not be working with the metadata 
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print([table[0] for table in tables])

cursor.execute("PRAGMA table_info(Per_Image);")
columns_info = cursor.fetchall()

# Extract just the column names
column_names = [col[1] for col in columns_info]
print(column_names)
quer = cursor.execute(
    "SELECT ImageNumber,Image_Metadata_WellRow,Image_Metadata_WellColumn,Image_Metadata_Field FROM Per_Image WHERE ImageNumber < 100;"
)
last = cursor.fetchall()
display(last)

['Per_Cell', 'Per_Cytoplasm', 'Per_Nuclei', 'Per_Image', 'Experiment', 'sqlite_sequence', 'Experiment_Properties', 'Per_Experiment']
['ImageNumber', 'Image_Count_AllNuclei', 'Image_Count_Cell', 'Image_Count_Cell_Masks', 'Image_Count_Cytoplasm', 'Image_Count_DilateCellMasks', 'Image_Count_ErodeCellMasks', 'Image_Count_LysosomePuncta', 'Image_Count_Lysosomes', 'Image_Count_MitoEndpoints', 'Image_Count_MitoEnds', 'Image_Count_Mitochondria', 'Image_Count_Mitochondria_Puncta', 'Image_Count_Multinucleate_Cells', 'Image_Count_Nuclei', 'Image_Count_Nuclei_Masks', 'Image_Count_OrigFilteredNuclei', 'Image_Count_ResizeCellMasks', 'Image_Count_ResizeNucleiMasks', 'Image_Count_SkeletonizedMitoObjects', 'Image_Count_UnfilteredLysosomes', 'Image_Count_UnfilteredMitochondria', 'Image_Count_ValidCells', 'Image_ExecutionTime_01Images', 'Image_ExecutionTime_02Metadata', 'Image_ExecutionTime_03NamesAndTypes', 'Image_ExecutionTime_04Groups', 'Image_ExecutionTime_05CorrectIlluminationCalculate', 'Image_Exec

[(1, '04', '01', '01'),
 (2, '04', '01', '02'),
 (3, '04', '01', '03'),
 (4, '04', '01', '04'),
 (5, '04', '01', '05'),
 (6, '02', '01', '06'),
 (8, '04', '01', '08'),
 (9, '04', '01', '09'),
 (10, '04', '01', '10'),
 (11, '04', '01', '11'),
 (12, '04', '01', '12'),
 (13, '02', '01', '13'),
 (14, '02', '01', '14'),
 (15, '02', '01', '15'),
 (16, '02', '01', '16'),
 (18, '02', '01', '18'),
 (19, '04', '01', '19'),
 (20, '02', '01', '20'),
 (21, '04', '01', '21'),
 (22, '04', '01', '22'),
 (23, '04', '01', '23'),
 (24, '04', '01', '24'),
 (25, '02', '01', '25'),
 (26, '04', '01', '26'),
 (28, '02', '01', '28'),
 (29, '04', '01', '29'),
 (30, '04', '01', '30'),
 (31, '04', '01', '31'),
 (32, '02', '01', '32'),
 (33, '04', '01', '33'),
 (34, '04', '01', '34'),
 (38, '04', '01', '38'),
 (39, '04', '01', '39'),
 (40, '04', '01', '40'),
 (41, '04', '06', '01'),
 (42, '04', '06', '02'),
 (43, '04', '06', '03'),
 (44, '04', '06', '04'),
 (45, '04', '06', '05'),
 (46, '04', '06', '06'),
 (47, '0

In [56]:
display(cell_df.shape,nucleus_df.shape,cyto_df.shape)

merged_df_nuc = pd.merge(
    combo_df,
    nucleus_df,
    how="left",
    left_on=["ImageNumber", "Cell_Number_Object_Number"],
    right_on=["ImageNumber", "Nuclei_Number_Object_Number"],
    
)

merged_df_nuc_2 = pd.merge(
    combo_df,
    nucleus_df,
    how="left",
    left_on=["Cell_Number_Object_Number","ImageNumber"],
    right_on=["Nuclei_Number_Object_Number","ImageNumber"],
)

merged_df_final = pd.merge(
    merged_df_nuc,
    cyto_df,
    how="left",
    left_on=["Cell_Number_Object_Number", "ImageNumber"],
    right_on=["Cytoplasm_Number_Object_Number", "ImageNumber"],
)
print(search_column_name(merged_df_final, "Area"))
merged_df_final["Cell_Unique_ID"] = merged_df_final.index
merged_df_final["Metadata_WellRowColumnField"] = "r" + merged_df_final["Metadata_WellRow"].astype(str) + "c" + merged_df_final["Metadata_WellColumn"].astype(str) + "f" + merged_df_final["Metadata_Field"].astype(str)
merged_df_final["Well_ImageNumber_CellNumber"] = merged_df_final["Metadata_Well"].astype(str) + "_" + merged_df_final["ImageNumber"].astype(str) + "_" + merged_df_final["Cell_Number_Object_Number"].astype(str)
merged_df_final["CellNumber_ImageNumber_Index"] = merged_df_final["Cell_Number_Object_Number"].astype(str)  + "_" + merged_df_final["ImageNumber"].astype(str) 

displaycols = [
    "ImageNumber",
    "Cell_Number_Object_Number",
    "Nuclei_Number_Object_Number",
    #"Cytoplasm_Number_Object_Number",
    "Cell_AreaShape_Area",
    "Nuclei_AreaShape_Area",
    #"Cytoplasm_AreaShape_Area",
    "Metadata_Well",
    #"Metadata_WellRowColumnField",
   # "Cell_Unique_ID",
]

display(merged_df_final[displaycols])
display(merged_df_nuc_2[displaycols])

(8313, 1203)

(8313, 776)

(8313, 890)

Query: Area
    Cell_AreaShape_Area
    Cell_AreaShape_BoundingBoxArea
    Cell_AreaShape_BoundingBoxMaximum_X
    Cell_AreaShape_BoundingBoxMaximum_Y
    Cell_AreaShape_BoundingBoxMinimum_X
    Cell_AreaShape_BoundingBoxMinimum_Y
    Cell_AreaShape_Center_X
    Cell_AreaShape_Center_Y
    Cell_AreaShape_Compactness
    Cell_AreaShape_ConvexArea
    Cell_AreaShape_Eccentricity
    Cell_AreaShape_EquivalentDiameter
    Cell_AreaShape_EulerNumber
    Cell_AreaShape_Extent
    Cell_AreaShape_FormFactor
    Cell_AreaShape_MajorAxisLength
    Cell_AreaShape_MaxFeretDiameter
    Cell_AreaShape_MaximumRadius
    Cell_AreaShape_MeanRadius
    Cell_AreaShape_MedianRadius
    Cell_AreaShape_MinFeretDiameter
    Cell_AreaShape_MinorAxisLength
    Cell_AreaShape_Orientation
    Cell_AreaShape_Perimeter
    Cell_AreaShape_Solidity
    Cell_AreaShape_Zernike_0_0
    Cell_AreaShape_Zernike_1_1
    Cell_AreaShape_Zernike_2_0
    Cell_AreaShape_Zernike_2_2
    Cell_AreaShape_Zernike_3_1
    Cell_AreaSh

,ImageNumber,Cell_Number_Object_Number,Nuclei_Number_Object_Number,Cell_AreaShape_Area,Nuclei_AreaShape_Area,Metadata_Well
0,6,1,1,159308.0,35168.0,B01
1,13,1,1,838467.0,104768.0,B01
2,14,1,1,97683.0,45244.0,B01
3,15,1,1,544332.0,72304.0,B01
4,16,1,1,324588.0,118072.0,B01
...,...,...,...,...,...,...
8308,1600,8,8,54568.0,20480.0,G11
8309,1600,9,9,88704.0,30736.0,G11
8310,1600,10,10,73676.0,19872.0,G11
8311,1600,11,11,83208.0,19328.0,G11


,ImageNumber,Cell_Number_Object_Number,Nuclei_Number_Object_Number,Cell_AreaShape_Area,Nuclei_AreaShape_Area,Metadata_Well
0,6,1,1,159308.0,35168.0,B01
1,13,1,1,838467.0,104768.0,B01
2,14,1,1,97683.0,45244.0,B01
3,15,1,1,544332.0,72304.0,B01
4,16,1,1,324588.0,118072.0,B01
...,...,...,...,...,...,...
8308,1600,8,8,54568.0,20480.0,G11
8309,1600,9,9,88704.0,30736.0,G11
8310,1600,10,10,73676.0,19872.0,G11
8311,1600,11,11,83208.0,19328.0,G11


In [ ]:
conn2 = sqlite3.connect(extra_db_path)
cursor = conn2.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print([table[0] for table in tables])

cursor.execute("PRAGMA table_info(Per_Lysosomes);")
columns_info = cursor.fetchall()
column_names = [col[1] for col in columns_info]
print(column_names)


lyso_df = pd.read_sql_query(
    "SELECT * FROM Per_Lysosomes WHERE ImageNumber < 10;", conn2
)  # Join on Lysosomes_Parent_Cell
mito_df = pd.read_sql_query(
    "SELECT * FROM Per_Mitochondria WHERE ImageNumber < 10;", conn2
)  # Join on Mitochondria_Parent_Cell
mito_ends_df = pd.read_sql_query(
    "SELECT * FROM Per_MitoEnds WHERE ImageNumber < 10;", conn2
)  # Join on Mitochondria_Parent_Cell

print(lyso_df.columns)
display(mito_ends_df)
file_path = "columnfeatures.txt"
with open(file_path, "w") as f:
    for col in cell_df.columns:
        f.write(col + "\n")



In [ ]:
def calculate_aggregated_object_features(
    parent_df,
    object_df,
    feature,
    parent_key,
    child_key="Cell_Number_Object_Number",
    aggregation="Median",
):
    """
    Calculate the aggregated function (typically median) of specified features grouped by a parent key.

    Parameters:
    df (DataFrame): The DataFrame containing the features.
    object_df (DataFrame): The DataFrame containing the object features.
    feature (str): The feature column to calculate the median for.
    child_key (str): The column name in the PARENT table that identifies the child obj.
    parent_key (str): The column name in the CHILD table that identifies the parent key.
    aggregation (str): the type of aggregation, can be "Mean","Median","Mode", or "Std"

    Returns:
    modified_df (DataFrame): The DataFrame with the new median feature column added.

    """
    agg_title = aggregation.title()  # make it title case for the column syntax
    if agg_title == "Median":
        agg_values = object_df.groupby([parent_key, "ImageNumber"])[feature].median()
    elif agg_title == "Mean" or agg_title == "Avg" or agg_title == "Average":
        agg_title = "Averaged"
        agg_values = object_df.groupby([parent_key, "ImageNumber"])[feature].mean()
    elif agg_title == "Sum":
        agg_title= "Total"
        agg_values = object_df.groupby([parent_key, "ImageNumber"])[feature].sum()
    elif agg_title == "Max":
        agg_values = object_df.groupby([parent_key, "ImageNumber"])[feature].max()
    elif agg_title == "Min":
        agg_values = object_df.groupby([parent_key, "ImageNumber"])[feature].min()
    elif agg_title == "Std":
        agg_values = object_df.groupby([parent_key, "ImageNumber"])[feature].std()
    else:
        ValueError('aggregation (str) not in "Mean","Median","Mode", or "Std"')
        return pd.DataFrame()
    
    #create the new column name for the aggregated feature
    col_name = f"Cell_{agg_title}_{feature}"
    
    #reformat the agg_values dataframe for merging
    agg_values = agg_values.reset_index()
    agg_values.columns = [child_key, 'ImageNumber', col_name]
    agg_values["CellNumber_ImageNumber_Index"] = (
        agg_values[child_key].astype(str) + "_" + agg_values["ImageNumber"].astype(str)
    )

    #Merge the aggregated values back into the parent dataframe
    modified_df = parent_df.copy()
    #add a column to parent_df to merge on
    modified_df["CellNumber_ImageNumber_Index"] = (
        modified_df[child_key].astype(str)
        + "_"
        + modified_df["ImageNumber"].astype(str)
    )

    modified_merged_df = modified_df.merge(
        agg_values[["CellNumber_ImageNumber_Index", col_name]],
        how="left",
        left_on="CellNumber_ImageNumber_Index",
        right_on="CellNumber_ImageNumber_Index",
    )

    return modified_merged_df

colnames = search_column_name(mito_df, "Image")

# Display columns containing 'metadata'
merged_df_mini = calculate_aggregated_object_features(
    merged_df_final[merged_df_final["ImageNumber"] < 10],
    mito_df,
    "Mitochondria_AreaShape_Area",
    parent_key="Mitochondria_Parent_Cell",
)

merged_df_mini = calculate_aggregated_object_features(
    merged_df_mini,
    mito_df,
    "Mitochondria_AreaShape_Area",
    parent_key="Mitochondria_Parent_Cell",
    aggregation="Mean"
)
merged_df_mini = calculate_aggregated_object_features(
    merged_df_mini,
    mito_df,
    "Mitochondria_AreaShape_Area",
    parent_key="Mitochondria_Parent_Cell",
    aggregation="Sum",
)

merged_df_mini = calculate_aggregated_object_features(
    merged_df_mini,
    mito_ends_df,
    "MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MitoSkeleton",
    parent_key="MitoEnds_Parent_Cell",
)

merged_df_mini["Cell_Mitochondria_Area_Occupied"] = merged_df_mini["Cell_Total_Mitochondria_AreaShape_Area"] / merged_df_mini["Cell_AreaShape_Area"]
merged_df_mini["Cell_Total_Mitochondria_AreaShape_Area_2"] = (
    merged_df_mini["Cell_Mean_Mitochondria_AreaShape_Area"]
    * merged_df_mini["Cell_Children_Mitochondria_Count"]
)

display(
    merged_df_mini[
        [
            "CellNumber_ImageNumber_Index",
            "Cell_Children_Mitochondria_Count",
            "Cell_Median_Mitochondria_AreaShape_Area",
            "Cell_Mean_Mitochondria_AreaShape_Area",
            "Cell_Averaged_Mitochondria_AreaShape_Area",
            "Cell_Total_Mitochondria_AreaShape_Area",
            "Cell_Mitochondria_Area_Occupied",
            "Cell_Total_Mitochondria_AreaShape_Area_2",
        ]
    ]
)
merged_df_mini["Cell_TotalMitoNonTrunkBranches"] = merged_df_mini[
    "Cell_Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn"
] * merged_df_mini["Cell_Children_MitoEnds_Count"]
fig, axs = plt.subplots(1, 4, sharey=True, sharex=True,figsize=(20, 5))
# sns.histplot(
#     merged_df_mini,
#     x="Cell_Mean_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MtSkltn",
#     color="g",
#     ax=axs[0],
# )
# sns.histplot(
#     merged_df_mini,
#     x="Cell_Median_MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MitoSkeleton",
#     color="c",
#     ax=axs[1],
# )
# sns.histplot(
#     merged_df_mini,
#     x="Nuclei_ObjectSkeleton_NumberNonTrunkBranches_MitoSkeleton",
#     color="y",
#     ax=axs[2],
# )
# sns.histplot(
#     mito_ends_df,
#     x="MitoEnds_ObjectSkeleton_NumberNonTrunkBranches_MitoSkeleton",
#     color="y",
#     ax=axs[3],
# )

sns.kdeplot(
    merged_df_mini, x="Cell_Mean_Mitochondria_AreaShape_Area", color="g", ax=axs[0]
)
sns.kdeplot(
    merged_df_mini, x="Cell_Median_Mitochondria_AreaShape_Area", color="c", ax=axs[1]
)
sns.kdeplot(
    merged_df_mini,
    x="Cell_Mean_Mitochondria_Puncta_AreaShape_Area",
    color="y",
    ax=axs[2],
)
sns.kdeplot(
    mito_df,
    x="Mitochondria_AreaShape_Area",
    color="m",
    ax=axs[3],
)

plt.ylim(top=0.005)
plt.xlim(0,5000)
#plt.ylim(0, 100)
plt.tight_layout()
plt.show()
# sns.kdeplot(mito_df, x="Mitochondria_AreaShape_Area")

In [7]:
root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v4_active/"
filename = "output.db"

db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
plate = "20250501_rep07"
combine_dfs = True
calculate_medians = True

plate_dfs = {}

pre_pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
# metadata extraction
map_file = os.path.join("plate_metadata", f"{plate}_metadata", "map.csv")
print(map_file)

pre_cell_df = pre_pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
# simplyfy the metadata column labels
pre_cell_df.columns = pre_cell_df.columns.str.replace(
    r"^Image_Metadata_", "Metadata_", regex=True
)
metadata_cols = [col for col in pre_cell_df.columns if "Metadata" in col]
# display(cell_df)

# merge the 1:1 dfs together
if combine_dfs:
    cell_df = combine_one_to_one_dfs(pre_cell_df, conn)

# Remove null/infinite rows
cols_to_check = ["Metadata_WellRow", "Metadata_WellColumn", "Metadata_Field"]
cell_df = cell_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN
cell_df = cell_df.dropna(subset=cols_to_check)  # Drop rows with NaN in these columns
# display(cell_df)

# add the median data
if calculate_medians:
    extra_feature_filename = "extra_features.db"
    extra_feature_db_path = os.path.join(root, extra_feature_filename)
    cell_df = load_organelle_medians(db_path=extra_feature_db_path, df=cell_df)
    cell_df.reset_index(drop=True)

# Find cell/nuc area ratio
cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(
    cell_df, "Cell_AreaShape_Area", "Nuclei_AreaShape_Area"
)

# get rid of cells where area is lower than nuc area
cell_df = cell_df[cell_df["Cell_Nuclei_Area_Ratio"] > 1]
# cell_df["Cell_Nuclei_Area_Ratio"].plot(kind="density",xlim=(-1,200))

display(
    cell_df[
        [
            "Cell_AreaShape_Area",
            "Nuclei_AreaShape_Area",
            "Cell_Nuclei_Area_Ratio",
            "Cell_Mean_Mitochondria_AreaShape_Area",
            "Cell_Median_Mitochondria_AreaShape_Area",
        ]
    ]
)

# make these metadatas string
cell_df["Metadata_WellRow"] = cell_df["Metadata_WellRow"].astype(int)
cell_df["Metadata_WellColumn"] = cell_df["Metadata_WellColumn"].astype(int)
cell_df["Metadata_Field"] = cell_df["Metadata_Field"].astype(int)
display(cell_df[metadata_cols])


print(os.path.exists(map_file))
if os.path.exists(map_file):
    platemap_df = pd.read_csv(map_file)
    display(platemap_df)

    platemap_df["Metadata_WellRow"] = platemap_df["Metadata_WellRow"].astype(int)
    platemap_df["Metadata_WellColumn"] = platemap_df["Metadata_WellColumn"].astype(int)
    platemap_df["Metadata_Field"] = platemap_df["Metadata_Field"].astype(int)
    # platemap_df.reset_index(drop=True)
    cell_df = cell_df.merge(
        platemap_df,
        on=["Metadata_WellRow", "Metadata_WellColumn", "Metadata_Field"],
        how="left",
    )
    # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    cell_df["Passage Group"] = cell_df["PassageNumber"].apply(passage_group)
    cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")

    # display(cell_df[cell_df["Metadata_Well"] == "A02"])

    cell_df = cell_df.merge(
        platemap_df,
        on=[
            "Metadata_WellRow",
            "Metadata_WellColumn",
            "Metadata_Field",
        ],
        how="left",
    )
    # display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])

    # additional metadatas
    cell_df["Metadata_WellRowColumnField"] = (
        "r"
        + cell_df["Metadata_WellRow"].astype(str)
        + "c"
        + cell_df["Metadata_WellColumn"].astype(str)
        + "f"
        + cell_df["Metadata_Field"].astype(str)
    )

    cell_df["Metadata_Plate"] = plate
    cell_df["Replicate_Number"] = plate[-1]
    cell_df["Replicate_String"] = f"R{plate[-1]}"

    cell_df["Replicate_WellRowColumnField"] = (
        cell_df["Replicate_String"] + "_" + cell_df["Metadata_WellRowColumnField"]
    )


min_x = 0
min_y = 0
max_x = cell_df["Image_Width_DAPI"][0]  # get the max x and y resolutions
max_y = cell_df["Image_Height_DAPI"][0]

cell_df_excluded_borders = exclude_borders(
    cell_df, min_x, min_y, max_x, max_y, prefix="Cell_"
)

plate_dfs[plate] = cell_df

combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
# display(combined_cell_df)


plate_metadata/20250501_rep07_metadata/map.csv


,Cell_AreaShape_Area,Nuclei_AreaShape_Area,Cell_Nuclei_Area_Ratio,Cell_Mean_Mitochondria_AreaShape_Area,Cell_Median_Mitochondria_AreaShape_Area
0,159308.0,35168.0,4.529914,81.405941,57.0
1,838467.0,104768.0,8.003083,109.683423,70.0
2,97683.0,45244.0,2.159027,106.176471,64.0
3,544332.0,72304.0,7.528380,120.840369,80.0
4,324588.0,118072.0,2.749068,135.971200,68.0
...,...,...,...,...,...
8308,54568.0,20480.0,2.664453,348.266667,102.0
8309,88704.0,30736.0,2.885997,253.145833,141.0
8310,73676.0,19872.0,3.707528,185.589744,83.0
8311,83208.0,19328.0,4.305050,166.949153,86.0


,Image_ExecutionTime_02Metadata,Metadata_Channels,Metadata_EmptyImage_Cells,Metadata_EmptyImage_Nuclei,Metadata_Field,Metadata_FileLocation,Metadata_Frame,Metadata_Series,Metadata_Well,Metadata_WellColumn,Metadata_WellRow
0,0.0,None,0,0,6,None,0,0,B01,1,2
1,0.0,None,0,0,13,None,0,0,B01,1,2
2,0.0,None,0,0,14,None,0,0,B01,1,2
3,0.0,None,0,0,15,None,0,0,B01,1,2
4,0.0,None,0,0,16,None,0,0,B01,1,2
...,...,...,...,...,...,...,...,...,...,...,...
8308,0.0,None,0,0,40,None,0,0,G11,11,7
8309,0.0,None,0,0,40,None,0,0,G11,11,7
8310,0.0,None,0,0,40,None,0,0,G11,11,7
8311,0.0,None,0,0,40,None,0,0,G11,11,7


True


,Metadata_Well,Metadata_WellRow,Metadata_WellColumn,Metadata_Field,TimepointName,SerialPassage_BatchNumber,AgeGroup,PassageNumber,Staining,Drug,FlaggedBatch
0,B01,2,1,1,SPB12 AG2_Doxo P12,12,2,12,LAMP1-488 + MitoRed + Phalloidin-647,Doxo,False
1,B01,2,1,2,SPB12 AG2_Doxo P12,12,2,12,LAMP1-488 + MitoRed + Phalloidin-647,Doxo,False
2,B01,2,1,3,SPB12 AG2_Doxo P12,12,2,12,LAMP1-488 + MitoRed + Phalloidin-647,Doxo,False
3,B01,2,1,4,SPB12 AG2_Doxo P12,12,2,12,LAMP1-488 + MitoRed + Phalloidin-647,Doxo,False
4,B01,2,1,5,SPB12 AG2_Doxo P12,12,2,12,LAMP1-488 + MitoRed + Phalloidin-647,Doxo,False
...,...,...,...,...,...,...,...,...,...,...,...
2875,G12,7,12,36,SPB13 AG0 P9,13,0,9,LAMP1-488 + MitoRed + Phalloidin-647,NaN,False
2876,G12,7,12,37,SPB13 AG0 P9,13,0,9,LAMP1-488 + MitoRed + Phalloidin-647,NaN,False
2877,G12,7,12,38,SPB13 AG0 P9,13,0,9,LAMP1-488 + MitoRed + Phalloidin-647,NaN,False
2878,G12,7,12,39,SPB13 AG0 P9,13,0,9,LAMP1-488 + MitoRed + Phalloidin-647,NaN,False


In [32]:
def pull_up_cp_segmentation_image_fromID(
    df,
    object_key,
    parent_dir="",
    replicate_col_name="Replicate_Number",
    feature="",
    image_channel="LAMP1",
    save=False,
    savepath="",
):
    """Function to pull up an image with cellprofiler segmentation outlines that has a matching image in the dataframe
    and also highlight the sepecific object with a bounding box

    Args:
        object_key (int): the uniqueobject key in the dataframe
        parent_dir (str, optional): _description_. Defaults to "~/".
        img_filename (str): _description_.
        replicate (int, optional): _description_. Defaults to 0.
        group (str, optional): _description_. Defaults to "".
    """
    from matplotlib import image as mpimg
    from PIL import Image

    # get the filename, replicate and coords from the unique ID if the ID exists
    if object_key is not None and df is not None:
        rect = get_object_bbox_coordinates_as_rectangle(df, object_key)

        unique_row = df[df["Cell_Unique_ID"] == object_key]
        img_filename = unique_row[f"Image_FileName_{image_channel}_MAX"].values[0]
        replicate = unique_row[replicate_col_name].values[0]
        passage = unique_row["PassageNumber"].values[0]
        group = unique_row["AllGroups"].values[0]
    else:
        print(f"Object number {object_key} not found in dataframe.")
        return False
    # loop over to find the file in the directory
    img_filename_noext = img_filename.split(".")[0]
    for root, dirs, files in os.walk(parent_dir):
        for filename in files:
            if (
                img_filename_noext.split("_")[1] in filename
                and filename.endswith(".png")
                and "active" in root  # active cp output folder
                and replicate == find_replicate_cp_output_folder(root)
            ):
                print(filename)
                img_path = os.path.join(
                    root,
                    filename,  # img_filename_noext + ".png"
                )  # make the path
                # Now we see if the image exists and try to open it
                try:
                    img = Image.open(img_path)
                    fig, ax = plt.subplots(figsize=(12, 12))
                    plt.imshow(img)
                    plt.axis("off")  # Turn off axis labels for a cleaner image display
                    plt.title(
                        f"ID:{object_key}, R{replicate} P{passage}, {filename}, {group}"
                    )
                    # Lets add a rectangle if the oject key is in the dataframe
                    ax.add_patch(rect)
                    # add a label with the feature value if it exists
                    if feature in df.columns:
                        feature_value = unique_row[feature].values[0]
                        plt.text(
                            rect.get_x(),
                            rect.get_y() - 10,
                            f"{feature}: {feature_value:.2f}",
                            color="lime",
                            fontsize=12,
                            weight="bold",
                            bbox=dict(facecolor="black", alpha=0.5, pad=2),
                        )
                    if save:
                        plt.savefig(
                            f"{savepath}/R{replicate}_P{passage}_{filename}_{object_key}.png",
                            bbox_inches="tight",
                        )
                    plt.show()
                    print(f"Segmented image url: {img_path}")
                    # img.show()
                    return True
                except FileNotFoundError:
                    print(
                        f"Image file {img_path} not found. Please ensure 'your_image.png' exists."
                    )
    return False


def get_object_bbox_coordinates_as_rectangle(
    df,
    unique_ID,
    coord_cols=[
        "Cell_AreaShape_BoundingBoxMinimum_X",
        "Cell_AreaShape_BoundingBoxMinimum_Y",
        "Cell_AreaShape_BoundingBoxMaximum_X",
        "Cell_AreaShape_BoundingBoxMaximum_Y",
    ]
):
    from matplotlib import patches

    row = df[df["Cell_Unique_ID"] == unique_ID]
    if not row.empty:
        x_min = row[coord_cols[0]].values[0]
        y_min = row[coord_cols[1]].values[0]
        x_max = row[coord_cols[2]].values[0]
        y_max = row[coord_cols[3]].values[0]
        width = x_max - x_min
        height = y_max - y_min
        rect = patches.Rectangle(
            (x_min, y_min),
            width,
            height,
            linewidth=2,
            edgecolor="red",
            facecolor="none",
        )
    return rect


def query_group_replicate_condition(
    df, group, replicate_number=0, condition_col="", op=operator.eq, value=None
):
    """
    Filter df by group, replicate_number, and a condition using a passed operator.
    Example: op=operator.lt for '<', op=operator.gt for '>', op=operator.eq for '=='
    """
    mask = (
        (df["AllGroups"] == group)
        & (df["Replicate_Number"] == replicate_number)
        & (op(df[condition_col], value))
    )
    return df[mask]

display(combined_cell_df[["ImageNumber","Cell_Number_Object_Number","Nuclei_Number_Object_Number","Cell_AreaShape_Area","Nuclei_AreaShape_Area"]])

,ImageNumber,Cell_Number_Object_Number,Nuclei_Number_Object_Number,Cell_AreaShape_Area,Nuclei_AreaShape_Area
0,6,1,1,159308.0,35168.0
1,13,1,1,838467.0,104768.0
2,14,1,1,97683.0,45244.0
3,15,1,1,544332.0,72304.0
4,16,1,1,324588.0,118072.0
...,...,...,...,...,...
8269,1600,8,8,54568.0,20480.0
8270,1600,9,9,88704.0,30736.0
8271,1600,10,10,73676.0,19872.0
8272,1600,11,11,83208.0,19328.0


In [30]:
import operator
feature_meas = "Nuclei_AreaShape_Area"
group = "P6-10"
condition_value = 0
display(
    query_group_replicate_condition(
        combined_cell_df,
        group,
        replicate_number=7,
        condition_col=feature_meas,
        op=operator.ge,
        value=condition_value,
    ).shape
)

combined_cell_df["Cell_Unique_ID"] = combined_cell_df.index
combined_cell_df.rename(columns={"PassageNumber_x": "PassageNumber"}, inplace=True)

valueslist = [
    "Cell_Unique_ID",
    "ImageNumber",
    "Metadata_WellRow",
    "Metadata_WellColumn",
    "Metadata_Field",
    "AllGroups",
    "Replicate_Number",
    "Cell_Number_Object_Number",
    "Cell_AreaShape_Area",
    "Nuclei_AreaShape_Area",
    "Cell_Nuclei_Area_Ratio",
    "Cell_Children_Mitochondria_Count",
    "Cell_Children_Lysosomes_Count",
    "Image_Width_DAPI",
    "Image_URL_MitoTracker_MAX",
    "Image_FileName_MitoTracker_MAX",
    "Image_FileName_LAMP1_MAX",
]

mini_filter_df = combined_cell_df[valueslist]
index_code = 0
# pull_up_cp_segmentation_image(
#     mini_filter_df["Image_FileName_LAMP1_MAX"][index_code],
#     "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output",
#     mini_filter_df["Replicate_Number"][index_code],
#     mini_filter_df["AllGroups"][index_code],
#     object_key=37170,
#     df=df_to_filter,
# )

object_key = 5

display(combined_cell_df[combined_cell_df["Cell_Unique_ID"] == object_key][valueslist])
savepath = f"/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/plots/curation/{feature_meas}"
os.makedirs(savepath, exist_ok=True)
for compartment in ["MitoTracker", "LAMP1"]:
    plot = pull_up_cp_segmentation_image_fromID(
        combined_cell_df,
        object_key=object_key,
        parent_dir="/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output",
        image_channel=compartment,
        feature=feature_meas,
        save=True,
        savepath=savepath,
    )
    print(plot)

(0, 3408)

,Cell_Unique_ID,ImageNumber,Metadata_WellRow,Metadata_WellColumn,Metadata_Field,AllGroups,Replicate_Number,Cell_Number_Object_Number,Cell_AreaShape_Area,Nuclei_AreaShape_Area,Cell_Nuclei_Area_Ratio,Cell_Children_Mitochondria_Count,Cell_Children_Lysosomes_Count,Image_Width_DAPI,Image_URL_MitoTracker_MAX,Image_FileName_MitoTracker_MAX,Image_FileName_LAMP1_MAX
5,5,16,2,1,16,Doxo,7,2,172140.0,33584.0,5.125655,160,132,2160,file:/mnt/bigdisk1/AllieSpangaro/Morphology_Re...,MAX_ch2-r02c01f16.tif,MAX_ch1-r02c01f16.tif


False
False


In [ ]:
def update_database_with_well_metadata(db_path):
    '''
        Update the database with well metadata.
        
        Args:
            db_path (str): Path to the database file
    '''
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()
    
    # Read Per_Image table
    image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
    
    # Add well metadata
    updated_image_df = add_well_metadata(image_df)
    
    # Write updated DataFrame back to the database
    try:
        updated_image_df.to_sql('Per_Image', conn, if_exists='replace', index=False)
        print("Database updated successfully with well metadata.")
    except Exception as e:
        print(f"Error updating database: {e}")
    #cursor.execute("SELECT Metadata_Well FROM Per_Image LIMIT 5;")
    #cursor.fetchall()
    conn.close()

#db_path = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v2_active/20250501_output.db"
#update_database_with_well_metadata(db_path)
    
def exclude_borders(df, min_x=0.0, min_y=0.0, max_x=2160.0, max_y=2160.0, prefix="Cell_"):
    """
    Exclude rows in the DataFrame that have bounding box coordinates outside the specified limits.
    
    Parameters:
    df (DataFrame): The DataFrame containing the bounding box coordinates.
    min_x (float): Minimum x-coordinate for exclusion.
    min_y (float): Minimum y-coordinate for exclusion.
    max_x (float): Maximum x-coordinate for exclusion.
    max_y (float): Maximum y-coordinate for exclusion.

    Returns:
    DataFrame: Filtered DataFrame with rows excluded based on bounding box coordinates.
    """
    filtered_df = df[
        (df[f"{prefix}AreaShape_BoundingBoxMaximum_X"] < max_x) &
        (df[f"{prefix}AreaShape_BoundingBoxMaximum_Y"] < max_y) &
        (df[f"{prefix}AreaShape_BoundingBoxMinimum_X"] > min_x) &
        (df[f"{prefix}AreaShape_BoundingBoxMinimum_Y"] > min_y)
    ]
    return filtered_df

def update_database_with_missing_values(db_to_update, mask_db, table_to_update ="Per_Cell", update_table="Per_Object"):
    '''
        Update the database with well metadata.
        
        Args:
            db_to_update (str): Path to the database file
            mask_db (str): The database containing the update
            
    '''
    conn = sqlite3.connect(mask_db)
    cursor = conn.cursor()
    
    # Read update table
    extra_df = pd.read_sql_query(f"SELECT * FROM {update_table}", conn)
    conn.close()
    
    conn = sqlite3.connect(db_to_update)
    existing_df = pd.read_sql_query(f"SELECT * FROM {table_to_update}", conn)
    rep5_image_numbers = existing_df["ImageNumber"]
    # Filter extra_data to exclude those ImageNumbers
    extra_data_mask = extra_df[~extra_df["ImageNumber"].isin(rep5_image_numbers)]
    display(extra_data_mask.shape)
    # Concatenate the filtered extra_data to the main dataframe
    updated_df = pd.concat([existing_df, extra_data_mask], ignore_index=True)
    #updated_df = extra_df
    
    display(updated_df.shape)
    display(updated_df.head())
    
    # Write updated DataFrame back to the database
    try:
        updated_df.to_sql('Per_Cell', conn, if_exists='replace', index=False)
        print("Database updated successfully with table values.")
    except Exception as e:
        print(f"Error updating database: {e}")
    #cursor.execute("SELECT Metadata_Well FROM Per_Image LIMIT 5;")
    #cursor.fetchall()
    conn.close()



### WARNING - only run db updates if you are 100% sure of your inputs

### Test metadata

## Get table info for a single plate

In [ ]:

plate_dfs = []

root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v3_active/"
filename = "20250501_output.db"
db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
plate = "20250501_rep07"
cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
tables = cursor.fetchall()
print([table[0] for table in tables])

cursor.execute("PRAGMA table_info(Per_Relationships);")
columns_info = cursor.fetchall()

# Extract just the column names
column_names = [col[1] for col in columns_info]
print(column_names)

#test_meta_df = pd.read_sql_query("SELECT * FROM Per_Image;", con=conn)
#display(test_meta_df)
test_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", con=conn)
test_cell_df_withmito = pd.read_sql_query("SELECT ImageNumber FROM Per_Cell WHERE Cell_Children_Mitochondria_Count > 0;", conn)
test_cell_df_withlyso = pd.read_sql_query(
    "SELECT ImageNumber FROM Per_Cell WHERE Cell_Children_Lysosomes_Count > 0;", conn
)
test_cell_df_withmitoandlyso = pd.read_sql_query(
    "SELECT ImageNumber FROM Per_Cell WHERE Cell_Children_Mitochondria_Count > 0 AND Cell_Children_Lysosomes_Count > 0;", conn
)
test_cell_df_withmitoandlysoandnuc = pd.read_sql_query(
    "SELECT ImageNumber FROM Per_Cell WHERE Cell_Children_Mitochondria_Count > 0 AND Cell_Children_Lysosomes_Count > 0 AND Cell_Children_Nuclei_Count > 0;",
    conn
)

test_mitocombo_df = pd.read_sql_query("SELECT * FROM Per_MergedMitoPerCell", conn)

test_lysocombo_df = pd.read_sql_query("SELECT * FROM Per_MergedLysoPerCell", conn)
#filtered_cell_df = enforce_objects_one_to_one(test_cell_df)
test_nuc_df = pd.read_sql_query("SELECT * FROM Per_Nuclei", conn)
test_nuccombo_df = pd.read_sql_query("SELECT * FROM Per_MergedNucleiPerCell", conn)
test_rel_df = pd.read_sql_query("SELECT * FROM Per_RelationshipTypes;", conn)

print(f"Shapes: Cell:{test_cell_df.shape} Nuclei: {test_nuc_df.shape} Combined Mito: {test_mitocombo_df.shape} Combined Lyso: {test_lysocombo_df.shape} Combined Nuc: {test_nuccombo_df.shape}")
print(f"Cells with at least one mito: {test_cell_df_withmito.shape}, cells with at least one lyso: {test_cell_df_withlyso.shape}, cells with at least one mito and lyso: {test_cell_df_withmitoandlyso.shape}, cells with at least one mito and lyso and nuc: {test_cell_df_withmitoandlysoandnuc.shape}")
display(test_rel_df)

In [ ]:
relationship_types = pd.read_sql_query("SELECT * FROM Per_RelationshipTypes", conn)
display(relationship_types)

#relationships = pd.read_sql_query("SELECT * FROM Per_Relationships", conn)


In [ ]:
root = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/20250501_rep07_output/v3_active/"
filename = "20250501_output.db"
db_path = os.path.join(root, filename)
conn = sqlite3.connect(db_path)
cursor = conn.cursor()
plate = "20250501_rep07"

plate_dfs = {}
nuclei_dfs = {}

pre_cell_df = pd.read_sql_query("SELECT * FROM Per_Cell", conn)
image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
#metadata extraction
map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
print(map_file)

cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
metadata_cols = [col for col in cell_df.columns if "Metadata" in col]
#display(cell_df)

#Remove null/infinite rows
cols_to_check = ['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field']
cell_df = cell_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN
cell_df = cell_df.dropna(subset=cols_to_check)        # Drop rows with NaN in these columns
#display(cell_df)

#Find cell/nuc area ratio
cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(cell_df, "Cell_AreaShape_Area")
#get rid of cells where area is lower than nuc area
cell_df = cell_df[cell_df["Cell_Nuclei_Area_Ratio"] > 1]
cell_df["Cell_Nuclei_Area_Ratio"].plot(kind="density",xlim=(-1,200))
display(
    cell_df[
        [
            "Cell_AreaShape_Area",
            "Cell_Mean_Nuclei_AreaShape_Area",
            "Cell_Children_Nuclei_Count",
            "Cell_Nuclei_Area_Ratio",
        ]
    ]
)

#make these metadatas string

cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
display(cell_df[metadata_cols])

cell_df.reset_index(drop=True)

#add the median data
#cell_df = load_organelle_medians(db_path, cell_df)

print(os.path.exists(map_file))
if os.path.exists(map_file):
    platemap_df = pd.read_csv(map_file)
    display(platemap_df)
    
    platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
    platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
    platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
    #platemap_df.reset_index(drop=True)
    cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
    #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
    cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
    cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
    
    #display(cell_df[cell_df["Metadata_Well"] == "A02"])
    
    cell_df["Metadata_Plate"] = plate
    cell_df["Replicate_Number"] = plate[-1]
    
min_x = 0
min_y = 0
max_x = cell_df["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = cell_df["Image_Height_DAPI"][0]
        
cell_df_excluded_borders = exclude_borders(cell_df, min_x, min_y, max_x, max_y, prefix="Cell_")

plate_dfs[plate] = cell_df

combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
#display(combined_cell_df)

In [ ]:
#exclude borders and make a ssubset of the data
debug_df = cell_df[[]]
test_df = cell_df[["ImageNumber", "Cell_Number_Object_Number",
				   "PassageNumber","Passage Group",
				   "Drug","AllGroups","Metadata_WellRow",
				   "Metadata_WellColumn","Metadata_Field",
       				"Cell_AreaShape_Area",
					"Cell_AreaShape_BoundingBoxArea",
				   "Cell_AreaShape_BoundingBoxMaximum_X",
				   "Cell_AreaShape_BoundingBoxMaximum_Y",
				   "Cell_AreaShape_BoundingBoxMinimum_X",
				   "Cell_AreaShape_BoundingBoxMinimum_Y",
				   "Cell_Median_Lysosomes_AreaShape_Area",
				   "Cell_Mean_Lysosomes_AreaShape_Area",
				   "Cell_Median_Mitochondria_AreaShape_Area",
				   "Cell_Mean_Mitochondria_AreaShape_Area",
				   "Cell_Median_Nuclei_AreaShape_Area",
				   "Cell_Mean_Nuclei_AreaShape_Area"]]

order =get_all_group_order()
test_df['AllGroups'] = pd.Categorical(test_df['AllGroups'], categories=order, ordered=True)

test_df_excluded_borders = exclude_borders(test_df, min_x, min_y, max_x, max_y, prefix="Cell_")
csv_dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"

test_df.head(1000).to_csv(os.path.join(csv_dir,"test_rep7.csv"))
test_df_excluded_borders.head(1000).to_csv( os.path.join(csv_dir,"test_rep7_borders_excluded.csv"))

test_df.describe().to_csv(os.path.join(csv_dir,"test_rep7_stats.csv"))
test_df_excluded_borders.describe().to_csv( os.path.join(csv_dir,"test_rep7_borders_excluded_stats.csv"))


test_df.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(os.path.join(csv_dir,"test_rep7_passage_group_stats.csv"))
test_df_excluded_borders.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(os.path.join(csv_dir,"test_rep7_excluded_borders_passage_group_stats.csv"))

fig, axes = plt.subplots(1, 2, figsize=(14, 6), sharey=True)
plt.style.use("ggplot")
sns.boxplot(
        x="AllGroups", y="Cell_AreaShape_Area", hue="AllGroups",
        palette="pastel", data=test_df, ax=axes[0],
        showfliers=False, width=0.9, order=order
    )
axes[0].set_title("Full Dataset")


    # Second subplot: Excluding Borders
sns.boxplot(
        x="AllGroups", y="Cell_AreaShape_Area", hue="AllGroups",
        palette="pastel", data=test_df_excluded_borders, ax=axes[1],
        showfliers=False, width=0.9, order=order
    )
axes[1].set_title("Excluding Cells Touching Borders")


plt.tight_layout()
    #plt.savefig(os.path.join(csv_dir, "rep7_cellsize_boxplots.png"))
plt.show()

# Exporting zone: 
## Loop over plates to check things
Make sure you put "_active" in the CP output folder names!

In [ ]:
# Setting file paths
curr_plates = ["20240313_rep01","20240326_rep02","20241018_rep03", "20241112_rep04", "20250328_rep05", "20250410_rep06", "20250501_rep07"] #"20240313_rep01_output","20240326_rep02_output"
parent_dir = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output"

parent_object_table = "Per_Cell"
compartment_tables = [ 'Per_MergedNucleiPerCell','Per_MergedMitoPerCell','Per_MergedLysoPerCell'] #note that these must be in a 1:1 relationship with cell

# query designed to remove cells from the dataset without any mitochondria, nuclei or lysosomes; allows us to have a 1:1 relationship
parent_obj_query = f"SELECT * FROM Per_{parent_object_table} WHERE Cell_Children_Mitochondria_Count > 0 AND Cell_Children_Lysosomes_Count > 0 AND Cell_Children_Nuclei_Count > 0;"

# Initialize a list to store the combined DataFrames
plate_dfs = {}
nuclei_dfs = {}


In [ ]:
def load_and_combine_plates_from_db(parent_dir,curr_plates,compartment_name= "Cell"):
# Loop over the plates
    for root, dirs, files in os.walk(parent_dir):
        for filename in files: 
            if filename.endswith(".db") and "active" in root and "extraRow1" not in filename:
                db_path = os.path.join(root, filename) #make the path
                for plate in curr_plates:
                    if plate in db_path:
                        conn = sqlite3.connect(db_path)
                        cursor = conn.cursor()
                        #update_database_with_well_metadata(db_path)
                        try:
                            # Read the 'Per_Cell' table and get metadata from 'Per_Image' table
                            pre_cell_df = pd.read_sql_query(f"SELECT * FROM Per_{compartment_name}", conn)
                            image_df = pd.read_sql_query("SELECT * FROM Per_Image", conn)
                            #metadata extraction - make sure its the exact same file format as the one above
                            map_file = os.path.join('plate_metadata', f"{plate}_metadata", "map.csv")
                            print(map_file)
                            
                            cell_df = pre_cell_df.merge(image_df, on=["ImageNumber"], how="left")
                            cell_df.columns = cell_df.columns.str.replace(r'^Image_Metadata_', 'Metadata_', regex=True)
                            
                            #remove rows where there isn't a valid row/column/field metadata
                            cols_to_check = ['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field']
                            cell_df = cell_df.replace([np.inf, -np.inf], np.nan)  # Replace inf with NaN
                            cell_df = cell_df.dropna(subset=cols_to_check)        # Drop rows with NaN in these columns
                            
                            #make these metadatas string
                            cell_df['Metadata_WellRow'] = cell_df['Metadata_WellRow'].astype(int)
                            cell_df['Metadata_WellColumn'] = cell_df['Metadata_WellColumn'].astype(int)
                            cell_df['Metadata_Field'] = cell_df['Metadata_Field'].astype(int)
                            
                            #add the median data
                            cell_df = load_organelle_medians(db_path, cell_df)
                            cell_df.reset_index(drop=True)
                            
                            #Find cell/nuc area ratio
                            cell_df["Cell_Nuclei_Area_Ratio"] = cell_nuc_area_ratio(cell_df, "Cell_AreaShape_Area")
                            #get rid of cells where area is lower than nuc area
                            cell_df = cell_df[cell_df["Cell_Nuclei_Area_Ratio"] > 1].reset_index(drop=True)
                            
                            print(os.path.exists(map_file))
                            
                            if os.path.exists(map_file):
                                platemap_df = pd.read_csv(map_file)
                                display(platemap_df)
                                platemap_df['Metadata_WellRow'] = platemap_df['Metadata_WellRow'].astype(int)
                                platemap_df['Metadata_WellColumn'] = platemap_df['Metadata_WellColumn'].astype(int)
                                platemap_df['Metadata_Field'] = platemap_df['Metadata_Field'].astype(int)
                                
                                #platemap_df.reset_index(drop=True)
                                cell_df = cell_df.merge(platemap_df, on=['Metadata_WellRow', 'Metadata_WellColumn', 'Metadata_Field'], how='left')
                                #display(cell_df[["Metadata_Field","Metadata_WellColumn","Metadata_WellRow","PassageNumber","Drug"]])
                                cell_df["Passage Group"] = cell_df['PassageNumber'].apply(passage_group)
                                cell_df["AllGroups"] = add_drug_to_group(cell_df, "Passage Group", "Drug")
                                
                                cell_df["Metadata_Plate"] = plate
                                cell_df["Replicate_Number"] = plate[-1]
                                
                            plate_dfs[plate] = cell_df
                        except Exception as e:
                            print(f"Error reading {db_path}: {e}")
                        finally:
                            conn.close()

    # Combine all DataFrames
    combined_cell_df = pd.concat(plate_dfs.values(), ignore_index=True)
    return combined_cell_df


In [ ]:
combined_cell_df = load_and_combine_plates_from_db(parent_dir,curr_plates)
#combined_nuclei_df = pd.concat(nuclei_dfs.values(), ignore_index=True)
                
#Filter DataFrames to only include cells that were stained with LAMP1-488 and MitoRed

#combined_nuclei_df_mitolyso = combined_nuclei_df[combined_nuclei_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]



## Export everything to CSV

In [ ]:
#Export to a giant csv
# filter out the non-experimental test images
combined_cell_df_mitolyso = combined_cell_df[combined_cell_df['Staining'].str.startswith("LAMP1-488 + MitoRed")]

display(combined_cell_df_mitolyso.head(10))
# print(cell_df.shape, " ", filter_df.shape)

#export the plate
outpath = "/mnt/bigdisk1/AllieSpangaro/Morphology_Replicative_Age_Project/CP_Output/postprocessed_csvs"
combined_cell_df_mitolyso.to_csv(
    os.path.join(outpath, "total_combined_cell.csv"), index=False
)

#set up the image borders and make a csv with excluded border cells
min_x = 0
min_y = 0
max_x = combined_cell_df_mitolyso["Image_Width_DAPI"][0] # get the max x and y resolutions
max_y = combined_cell_df_mitolyso["Image_Height_DAPI"][0]

combined_cell_df_mitolyso_borders_excluded = exclude_borders(
    combined_cell_df_mitolyso, min_x, min_y, max_x, max_y, prefix="Cell_"
)
combined_cell_df_mitolyso_borders_excluded.to_csv(
    os.path.join(outpath, "total_combined_cell_borders_excluded.csv"), index=False
)


### Summary Stats

In [ ]:
#Make summary stats for the csvs
combined_cell_df_mitolyso.describe().to_csv(
    os.path.join(outpath, "total_combined_cell_stats.csv")
)
combined_cell_df_mitolyso.groupby("AllGroups")["Cell_AreaShape_Area"].describe().to_csv(
    os.path.join(outpath, "total_combined_cell_passage_group_stats.csv")
)

combined_cell_df_mitolyso_borders_excluded.describe().to_csv(
    os.path.join(outpath, "total_combined_cell_borders_excluded_stats.csv")
)
combined_cell_df_mitolyso_borders_excluded.groupby("AllGroups")[
    "Cell_AreaShape_Area"
].describe().to_csv(
    os.path.join(
        outpath, "total_combined_cell_borders_excluded_passage_group_stats.csv"
    )
)


# Define the cell features
- note: stop here if you are doing more processing, import the exported csv instead to save time/RAM

In [ ]:
#Add extra columns 
def proportion_area_occupied_per_cell(df, compartment):
    #proportion of area occupied = children * mean organelle area / cell area
    colname = 'Total_Area_Proportion_' + compartment + '_Per_Cell'
    
    #children = 'Children_' + compartment + '_Count'
    #mean_organelle_area = 'Mean_'+ compartment + '_AreaShape_Area'
    organelle_area = compartment + '_AreaShape_Area'
    cell_area = 'AreaShape_Area'
    #df[colname] = df.apply(lambda x: (x[children] * x[mean_organelle_area]) / x[cell_area], axis=1)
    df[colname] = df.apply(lambda x: (x[organelle_area]) / x[cell_area], axis=1)
    return df[colname]

def mean_intesity_per_compartment_per_cell(df,compartment, name, tag):
    # Calculate the mean intensity of each compartment per cell
    #mean_intesity_per_compartment = integrated / (children*mean_area)
    colname = 'MeanIntensity_Per_' + compartment + '_Per_Cell'
    integrated = 'Intensity_IntegratedIntensity_' + tag
    #children = 'Children_' + compartment + '_Count'
    #mean_area = 'Mean_'+ compartment + '_AreaShape_Area'
    total_organelle_area = name + '_AreaShape_Area'
    df[colname] = df.apply(lambda x: x[integrated] / x[total_organelle_area], axis=1)
    return df[colname]


combined_cell_df_mitolyso_merged["Mean_Mitochondria_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(combined_cell_df_mitolyso_merged, "MergedMitoPerCell")
combined_cell_df_mitolyso_merged["Mean_Lysosomes_Area_PerCell_Ratio"]  = proportion_area_occupied_per_cell(combined_cell_df_mitolyso_merged, "MergedLysoPerCell")

combined_cell_df_mitolyso_merged["MeanIntensity_Lysosomes_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(combined_cell_df_mitolyso_merged, "Lysosomes", "MergedLysoPerCell", "LAMP1")
combined_cell_df_mitolyso_merged["MeanIntensity_Mitochondria_PerCell_Ratio"] = mean_intesity_per_compartment_per_cell(combined_cell_df_mitolyso_merged, "Mitochondria", "MergedMitoPerCell", "MitoTracker")


display(combined_cell_df_mitolyso_merged[["Passage Group","Mean_Lysosomes_Intensity_MeanIntensity_LAMP1","MeanIntensity_Mitochondria_PerCell_Ratio","MeanIntensity_Lysosomes_PerCell_Ratio","Mean_Lysosomes_Area_PerCell_Ratio","Mean_Mitochondria_Area_PerCell_Ratio","Math_Total_Mitochondria_AreaShape_Area_PerCell","Math_Total_Lysosomes_AreaShape_Area_PerCell"]])

In [ ]:

#file_path = '/mnt/bigdisk1/AllieSpangaro/HTP_fibroblasts_mitolyso/Cellcsv_columns.txt'


columns_list = define_cell_features(combined_cell_df_mitolyso_merged)

mito_features = make_feature_dict([col for col in columns_list if 'Mito' in col])
lyso_features = make_feature_dict([col for col in columns_list if 'Lysosome' in col or 'LAMP1' in col or 'Lyso' in col])
nuc_features = make_feature_dict([col for col in columns_list if 'Nuc' in col or 'DAPI' in col])
print(columns_list)


## Feature lists here:

In [ ]:
feature_dicts = [mito_features, lyso_features, nuc_features]
feature_names = ["Mitochondria Features", "Lysosome Features", "Nucleus Features"]

# Define the output file path
output_file_path = 'allfeatures_file.md'

# Open the file in write mode
with open(output_file_path, 'w') as file:
    for feature_name, feature_dict in zip(feature_names, feature_dicts):
        file.write(f"# {feature_name}\n\n")
        for feature_type, features in feature_dict.items():
            file.write(f"## {feature_type.capitalize()}\n")
            for feature in features:
                file.write(f"- {feature}\n")
            file.write("\n")
            
print(f"List has been written to {output_file_path}")


sns.pairplot(cell_df, hue='Passage Group', vars=mito_features['radialdistribution'], diag_kind='kde', plot_kws={'alpha':0.5})
plt.show()

# Functions for Data Analysis
calulcate normalizations, remove extreme left outliers, etc

## Normalize features to control (Passage 6-8)

In [ ]:
norm_cell_df = combined_cell_df_mitolyso_merged.copy()
norm_cell_df_nuc = apply_feature_normalization(norm_cell_df, nuc_features, curr_plates).copy()
norm_cell_df_mito = apply_feature_normalization(norm_cell_df_nuc, mito_features, curr_plates).copy()
norm_cell_df_mitolyso = apply_feature_normalization(norm_cell_df_mito, lyso_features, curr_plates)

#watch out - merged df might be clipping off all of the features


## Trying out the pivot and groupby functions


In [ ]:
#average_grouped_df = average_groups_by_plate(norm_cell_df_mitolyso, x_value='Passage Group', y_value='Intensity_MeanIntensity_MitoTracker', replicates='Replicate_Number')
#average_grouped_df_pivot = average_groups_pivot(average_grouped_df, x_value='Passage Group', y_value='Intensity_MeanIntensity_MitoTracker', replicates='Replicate_Number')
#display(average_grouped_df)
#display(average_grouped_df_pivot)


In [ ]:
categorical_col = 'Passage Group'
#order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21', 'P22-24']
df_test = norm_cell_df_mitolyso.copy()

pairs = getpairs(df_test, categorical_col, order)

# Print the pairs
print(pairs)

## It's plotting time


In [ ]:
def average_groups_by_plate_1(df, x_value, y_value, replicates):
    '''
    Group the DataFrame by the specified columns and calculate the mean of the y_value column.
    Returns the averaged dataframe for plotting
    '''
    df = df.dropna(subset=[x_value, y_value, replicates])
    df = df[df[y_value] != 0]

    df.reset_index(drop=True, inplace=True)
    
    group_averages = df.groupby([x_value, replicates], as_index=False, observed=True).agg({y_value: "mean"})
    
    # Reset the index to get a clean DataFrame
    average_df = group_averages.reset_index()
   
    return average_df

def make_single_feature_df_1(data, group, feature, replicates):
  pd.options.mode.copy_on_write = True
  
  subset=[group, feature, replicates]
  
  df = data.dropna(subset = subset).reset_index(drop=True)
  df = df[df[feature] != 0]
  
  df_subset = df[subset]
  df_subset[group] = df[group].astype('category')
  df_subset.reset_index(drop = True, inplace = True)
  
  return df_subset 

def oneway_anova(data, group_name, feature_meas):
  from scipy.stats import f_oneway

  data = data.dropna(subset=[group_name, feature_meas])
  data = data[data[feature_meas] != 0]
  
  groups = data[group_name].unique()
  data = [data[data[group_name] == group][feature_meas].dropna() for group in groups]
  anova_result = f_oneway(*data)
  
  print(f"ANOVA F-statistic: {anova_result.statistic}, ANOVA p-value: {anova_result.pvalue}")
  return anova_result


In [ ]:
feature_meas = 'Children_Lysosomes_Count'
pairs = getpairs(norm_cell_df_mitolyso, 'Passage Group', order)

feature_df = make_single_feature_df_1(norm_cell_df_mitolyso, group='Passage Group', feature=feature_meas, replicates='Replicate_Number')
group_avg_df = average_groups_by_plate_1(feature_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')
group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')

display(feature_df)
display(group_avg_df)
display(group_avg_df_pivot)

sns.set_theme(style="ticks")
#sns.set_context("notebook", font_scale=1.9)

plt.figure(figsize=(12, 8))

sns.violinplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Passage Group",
            order=order,
            fill = False,
            palette = 'pastel',
            cut=0,
            linecolor= 'k',
            inner_kws=dict(box_width = 5))

ax = sns.swarmplot(data=group_avg_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            palette='Set2',
            size=10, 
            edgecolor="k", 
            linewidth=1,
            dodge=False)

sns.pointplot(data=group_avg_df, x='Passage Group',
              y=feature_meas,
              #hue='Replicate_Number',
              color='dimgray',
              order=order,
              dodge=False,
              markers='_',
              linestyle=None,
              errorbar=None,
              ax=ax)


ax.legend_.remove()

sns.despine()
plt.gcf()#.set_size_inches(10, 6)
plt.xlabel('Passage Group')
plt.ylabel(feature_meas.replace('_',' '))

from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

# Extract the data for each group

# Perform the one-way ANOVA test

# Print the results
anova = oneway_anova(group_avg_df, 'Passage Group', feature_meas)

tukey_results = tukey_test_1(group_avg_df, test_groups='Passage Group', feature=feature_meas)
display(tukey_results)

annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order)
annotator.configure(text_format='star', loc='inside', verbose = 2)
annotator.set_pvalues_and_annotate(tukey_results['p-value'])

#plt.savefig(feature_meas + '_superviolinplot.png', dpi=300)
plt.show()



#### For Kruskal-Walis
```python
sns.catplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            fill = False,
            palette='Set2',
            kind = 'violin',
            inner = None)

sns.boxplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            showfliers = False,
            palette='Set2')

ax = sns.swarmplot(data=group_avg_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order)

sns.despine()
plt.gcf().set_size_inches(10, 6)
plt.xlabel('Passage Group')
plt.ylabel(feature_meas.replace('_',' '))
plt.ylim(-0.5, 6)


from statannotations.Annotator import Annotator
from statannotations.stats.StatTest import StatTest

annotator = Annotator(ax, pairs, data=group_avg_df_pivot, x='Passage Group', y=feature_meas, order=order) 0 
annotator.configure(test='Kruskal', text_format='star', loc='inside')
annotator.apply_and_annotate()


plt.savefig(feature_meas + '_boxplot.png', dpi=300)
```


## Make the plots - nonparametric
### ytitles
ytitle = "Distance of Mitochondria from Cell Center"
ytitle = "Distance of Lysosomes from Cell Center"
"Number of Mitochondria per Cell (Relative to Control)"
ytitle = "Number of Lysosomes per Cell (Relative to Control)"
ytitle = "Total Normalized Mitochondrial Area Per Cell"
ytitle = "Normalized # of Mitochondrial Endpoints
ytitle = "Normalized # of Mitochondrial Trunks"
ytitle = "Mean Length of Mitochondrial Network"
ytitle = "Mean # of Mitochondrial Branches"
ytitle = "Minimum Distance of Mitochondria from Cell"
ytitle = "Mean Total Mitochondrial Area"
ytitle = "Mean Mitochondrial Size"
ytitle = "Mean Total Mitochondrial Perimeter"
ytitle = "Mean Mitochondrial Granularity"
ytitle = "Total Mitochondrial Intensity"
ytitle = "Mean LAMP1 Intensity Per Lysosome"
ytitle = "Mean Mitochondria Intensity Per Cell"
ytitle = "Mean Lysosome Intensity Per Cell"
ytitle = "Total LAMP1 Intensity"
ytitle = "Total Mitochondrial Intensity"
ytitle = "Mean MitoTracker Intensity"
ytitle = "Ratio of Mitochondria Area to Cell Area"
ytitle = "Ratio of Lysosome Area to Cell Area"
ytitle = "MitoTracker Intensity Per Cell Area"
ytitle = "LAMP1 Intensity Per Cell Area"
ytitle = "Lysosomal Circularity"
ytitle = "MitoTracker Intensity Per Cell Area"

In [ ]:
#Nonparametric Function version
feature_meas = 'Mean_Lysosomes_Area_PerCell_Ratio'
order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21']#, 'P22-24']

if feature_meas == 'AreaShape_Area':
    norm_cell_df_mitolyso[feature_meas] = normalize_to_control(norm_cell_df_mitolyso,feature_meas)
    ytitle = "Mean Cell Size"


ytitle = "Mean Lysosome Area Per Cell Area"

ylimit = None# (-1,2)

pallete = "deep"
#pallete = "pastel" 
remove_outliers = True

def remove_outliers_iqr(df, col = None):
    cols = df.select_dtypes('number').columns  # limits to a (float), b (int) and e (timedelta)
    df_sub = df.loc[:, cols]

    iqr = df_sub.quantile(0.75, numeric_only=False) - df_sub.quantile(0.25, numeric_only=False)
    
    #calculate  extreme outlisers by dividing median by iqr
    lim = np.abs((df_sub - df_sub.median()) / iqr) < 2.22

    # replace outliers with nan
    df.loc[:, cols] = df_sub.where(lim, np.nan)
    df.dropna(subset=cols, inplace=True) # drop rows with NaN in numerical columns
    return df

def make_superplot_with_kruskal(data, group, feature_meas, replicates, ytitle = None, pallete='pastel', ylim = None, remove_outliers = remove_outliers):
    order = get_all_group_order()
    
    if ytitle is None:
        ytitle = feature_meas.replace('_', ' ')

       
    feature_df = make_single_feature_df(data, group=group, feature=feature_meas, replicates=replicates)
     
    if remove_outliers is True:
        feature_df = remove_outliers_iqr(feature_df)
        display(feature_df)
        
    
    pairs = getpairs(feature_df, group, order)

    #Remove the n=1 replicate
    feature_df = feature_df[feature_df[group] != "P22-24"]


    group_avg_df = average_groups_by_plate(feature_df, x_value=group, y_value=feature_meas, replicates=replicates)
    group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value=group, y_value=feature_meas, replicates=replicates)

    sns.set_theme(style="ticks")
    sns.set_context("talk", font_scale=0.5)

    plt.figure(dpi=300)

    sns.violinplot(data=feature_df, x=group,
                y=feature_meas,
                order=order,
                fill = False,
                color= 'gainsboro',
                cut=1,
                native_scale=True,
                linecolor='k',
                inner= None,
                #inner_kws=dict(box_width = 5)
                )

    ax = sns.swarmplot(data=group_avg_df, x=group,
                y=feature_meas,
                hue = replicates,
                order=order,
                palette=pallete,
                size=10, 
                edgecolor="k", 
                linewidth=1,
                dodge=0.5)
    
    #use a boxplot to draw the mean line - thinking outside the box :)
    sns.boxplot(data = group_avg_df, x = group,
                y = feature_meas,
                showmeans=True,
                meanline=True,
                meanprops={'color': 'dimgray', 'ls': '-', 'lw': 2.5},
                medianprops={'visible': False},
                whiskerprops={'visible': False},
                zorder=1,
                showfliers=False,
                showbox=False,
                showcaps=False,
                ax = ax)

    ax.legend_.remove()

    sns.despine()
    plt.gcf()#.set_size_inches(10, 6)
    plt.xlabel(group)
    plt.ylabel(ytitle)
    plt.ylim(ylim)

    from statannotations.Annotator import Annotator
    annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order) 
    annotator.configure(test='Kruskal', 
                        text_format='star', 
                        loc='inside', 
                        hide_non_significant = True,
                        color = 'black',
                        verbose = 2)
    annotator.apply_and_annotate()

    plt.savefig(feature_meas + '_superviolinplot.png', dpi=300)
    plt.show()
    


make_superplot_with_kruskal(norm_cell_df_mitolyso, 'Passage Group', feature_meas, 'Replicate_Number', ytitle, pallete, ylimit)

In [ ]:
#Nonparametric version
feature_meas = 'Children_Lysosomes_Count'
order = ['P6-8', 'P9-10', 'P11-13', 'P14-16', 'P17-18', 'P20-21']#, 'P22-24']

feature_df = make_single_feature_df_1(norm_cell_df_mitolyso, group='Passage Group', feature=feature_meas, replicates='Replicate_Number')
pairs = getpairs(feature_df, 'Passage Group', order)

#Remove the n=1 replicate
feature_df = feature_df[feature_df['Passage Group'] != "P22-24"]

group_avg_df = average_groups_by_plate_1(feature_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')
group_avg_df_pivot = average_groups_pivot(group_avg_df, x_value='Passage Group', y_value=feature_meas, replicates='Replicate_Number')

sns.set_theme(style="ticks")
sns.set_context("talk", font_scale=0.7)

plt.figure(dpi=300)

sns.violinplot(data=feature_df, x='Passage Group',
            y=feature_meas,
            #hue = "Passage Group",
            order=order,
            fill = False,
            color= 'gainsboro',
            #palette = 'Set2',
            cut=2,
            native_scale=True,
            linecolor='k',
            inner= None,
            #inner_kws=dict(box_width = 5)
            )

ax = sns.swarmplot(data=group_avg_df, x='Passage Group',
            y=feature_meas,
            hue = "Replicate_Number",
            order=order,
            palette='pastel',
            size=10, 
            edgecolor="k", 
            linewidth=1,
            dodge=0.5)

sns.pointplot(data=group_avg_df, x='Passage Group',
              y=feature_meas,
              #hue='Replicate_Number',
              color='dimgray',
              order=order,
              dodge=False,
              markers='_',
              linestyle=None,
              errorbar=None,
              ax=ax)

ax.legend_.remove()

sns.despine()
plt.gcf()#.set_size_inches(10, 6)
plt.xlabel('Passage Group')
plt.ylabel(feature_meas.replace('_',' '))
plt.ylim(-1,11)

from statannotations.Annotator import Annotator
annotator = Annotator(ax, pairs, data=group_avg_df_pivot, order=order) 
annotator.configure(test='Kruskal', 
                    text_format='star', 
                    loc='inside', 
                    hide_non_significant = True,
                    color = 'black',
                    verbose = 2)
annotator.apply_and_annotate()

plt.savefig(feature_meas + '_superviolinplot.png', dpi=300)
plt.show()

### To export the normalized csv:


In [ ]:

norm_cell_df_mitolyso.to_csv(os.path.join('All_Cell_w_metadata_normalized.csv'), index=False)

# Plotly Functions
## Make the plots and validate dist

In [ ]:
def make_layout(xtitle,ytitle):
  design = go.Layout(
        plot_bgcolor="#FFF",
        xaxis=dict(
            title=xtitle,
            linecolor="black",
            showgrid=False,
            titlefont=dict(size=20),
            tickfont=dict(size=16, color="black")
        ),
        yaxis=dict(
            title=ytitle,
            linecolor="black",
            showgrid=False,
            titlefont=dict(size=20),
            tickfont=dict(size=16, color="black")
        ),
        font=dict(size=14),
        legend=dict(
            title="",
            itemsizing='constant',
            font=dict(size=16, color="black"),
            tracegroupgap=10,
            traceorder='normal',
            orientation="h",
            yanchor="bottom",
            y=1.02,
            xanchor="right",
            x=1
        ),
        boxgap=0.4,
        boxgroupgap=0.05,
        width=900,  # Specify width of the plot
        height=600
    )
  return design

def box_param(fig, color, design):
  fig.update_traces(boxmean=True)
  fig.update_traces(jitter=1.0)
  fig.update_traces(boxpoints=False)
  fig.update_layout(design)
  fig.update_traces({'opacity': 0.9})
  fig.update_traces(marker_color=color)
  return fig

In [ ]:
#define parameter for plotly boxplot
def boxplot(df, variable, ytitle, xtitle, box_color, save=False, save_name=None):
    
    # #some styling stuff
  layout = make_layout(xtitle,ytitle)
  temp_copy = df.copy()
  temp_copy = outlier_removal(temp_copy, variable)
  temp_copy2 = normalize_to_control(temp_copy, variable)

  fig = px.box(temp_copy2, x="Passage Group", y=variable)
  fig = box_param(fig, box_color, layout)
  fig.update_traces(quartilemethod="inclusive")
  fig.show()
  
  if save==True:
      fig.write_image(save_name)

##  Call plotly
```python
boxplot(df, # dataframe name: mito_df, nuclei_df, image_df, outline_df, lysosomes_df
        'Variable', # the variable you wanna plot, column name
        'Y Title',# name of your y-axis (custom)
        'Time Point',#name of your x-axis (custom)
        'Red', # color of the boxes
        save=True, # if wanna save change to False to True, False is default
        save_name='_boxplot.png') #specify the name of the plot that you save

```

In [ ]:
#testing
pd.options.mode.copy_on_write = True
boxplot(cell_df,
            'Texture_Contrast_MitoTracker_3_01_256',
            'Texture_Contrast_MitoTracker_3_01_256',
            "Time Point",
            'Red',
            save = False,
            save_name = "_boxplot.png")

In [ ]:
#save all the files for that one feature

colors_i = 0
compartment = 'Mito'
#import kaleido

def save_all_single_feature_plots(data, features):
    '''
    Go through the data given and make plots
    Return: nothing; just outputs the plots
    '''
    for feature in features:
        if colors_i >= len(px.colors.qualitative.Dark24):
            colors_i = 0
        boxplot(data,
                feature,
                feature.replace('_', ' '),
                "Time Point",
                px.colors.qualitative.Dark24[colors_i],
                save = True,
                save_name = plate + "_" + compartment + "_" + feature + "_boxplot.png")
        colors_i = colors_i + 1
    
            
    

## The clustering zone

In [ ]:
from sklearn.decomposition import PCA
filter_key = 'LAMP1'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

pca = PCA()
components = pca.fit_transform(X)
labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
}

fig = px.scatter_matrix(
    components,
    labels=labels,
    dimensions=range(4),
    color=ordered_cell["Time"]
)
fig.update_traces(diagonal_visible=False)
fig.show()

In [ ]:
from sklearn.decomposition import PCA
filter_key = '_MitoTracker'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

pca = PCA(n_components=3)
components = pca.fit_transform(X)

fig = px.scatter_3d(components, x=0, y=1,z=2, color=ordered_cell['Time'] , labels = {
    str(i): f"PC {i+1} ({var:.1f}%)"
    for i, var in enumerate(pca.explained_variance_ratio_ * 100)
})
fig.show()

In [ ]:
from sklearn.manifold import TSNE
filter_key = 'Entropy_LAMP1'
filtered_features = list(filter(lambda x: filter_key in x,cell_features))

ordered_cell = cell_df.sort_values(
  by='Time', 
  ascending=True)
ordered_cell['Time']=ordered_cell['Time'].astype("string")
X = ordered_cell[filtered_features]

tsne = TSNE(n_components=2, random_state=0)
projections = tsne.fit_transform(X)

fig = px.scatter(
    projections, x=0, y=1,
    color=ordered_cell['Time']
)
fig.show()

In [ ]:
ordered_nuc = combined_nuclei_df.sort_values(
  by='Time', 
  ascending=True)

ordered_nuc['Passage Group'] = ordered_nuc['PassageNumber'].apply(passage_group)

feature = 'AreaShape_Solidity'

fig = px.box(ordered_nuc, x=feature, y='Passage Group', color = 'Passage Group', labels={
                     feature: feature.strip('_'),
                     'Passage Group': 'Passage Group'})

fig.show()